In [ ]:
import pandas as pd

# Configuration

In [ ]:
# Auto-reload the custom package
%load_ext autoreload
%autoreload 1
%aimport fairness_multiverse


In [ ]:
# If you want to force a specific run, set this to e.g. "41"
# If left as None, we will read the latest run number from output/counter.txt
RUN_TO_ANALYSE = None

from pathlib import Path

if RUN_TO_ANALYSE is None:
    counter_file = Path("output") / "counter.txt"
    if counter_file.is_file():
        with open(counter_file, "r") as fp:
            RUN_TO_ANALYSE = fp.read().strip()
    else:
        raise FileNotFoundError(
            "output/counter.txt not found. "
            "Run multiverse_analysis.py at least once, "
            "or set RUN_TO_ANALYSE manually (e.g. '41')."
        )

print(f"Using RUN_TO_ANALYSE = {RUN_TO_ANALYSE}")

In [ ]:
PREFIX_SETTINGS = "sett_"
PREFIX_EVAL = "sett_eval_"
PREFIX_PERFORMANCE = "perf_"
PREFIX_FAIRNESS = "fair_"

In [ ]:
from pathlib import Path

OUTPUT_DIR = Path(".") / "output"

# Directory that will contain outputs from analysis
ANALYSIS_OUTPUT_DIR = OUTPUT_DIR / "analyses" / (str(RUN_TO_ANALYSE))
ANALYSIS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = OUTPUT_DIR / "runs" / str(RUN_TO_ANALYSE) / "data"

In [ ]:
main_fairness_metric = "fair_main_equalized_odds_difference"

## Data Loading

In [ ]:
df_agg_raw = pd.read_csv(DATA_DIR / f"agg_{RUN_TO_ANALYSE}_run_outputs.csv.gz")

In [ ]:
import json

df_settings = pd.json_normalize(
    df_agg_raw["universe_settings"].apply(json.loads)
).add_prefix(
    PREFIX_SETTINGS
)

df_agg_full = df_settings.join(df_agg_raw)
df_agg_full.head()

In [ ]:
rows, columns = df_agg_full.shape
print(f"The data has N = {rows} rows and N = {columns} columns.")

In [ ]:
cols_settings = list(df_agg_full.columns[df_agg_full.columns.str.startswith(PREFIX_SETTINGS)])
cols_eval = list(df_agg_full.columns[df_agg_full.columns.str.startswith(PREFIX_EVAL)])
cols_non_eval = list(set(cols_settings) - set(cols_eval))
cols_performance = list(df_agg_full.columns[df_agg_full.columns.str.startswith(PREFIX_PERFORMANCE)])
cols_fairness = list(df_agg_full.columns[df_agg_full.columns.str.startswith(PREFIX_FAIRNESS)])


In [ ]:
df_agg_full["universe_id"]

In [ ]:
drop_mask = df_agg_full["sett_exclude_subgroups"].isin(['keep-largest_race_1'])
print(f"Dropping N = {drop_mask.sum()} rows, keeping N = {(~drop_mask).sum()}")

df_agg_full = df_agg_full[~drop_mask]

In [ ]:
df_agg_full["sett_exclude_subgroups"].unique()

In [ ]:
df_agg_full["sett_eval_fairness_grouping"].unique()

In [ ]:
df_agg_full["sett_eval_exclude_subgroups"].unique()

In [ ]:
df_agg_full["sett_eval_on_subset"].unique()